**Imports and setup**, including `utils` (spatial smoothing) and the hsiViewer.

In [ ]:
from sklearn import linear_model
import matplotlib.pyplot as plt
from matplotlib import colors
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
import numpy as np
from sklearn.mixture import GaussianMixture
import numpy as np
import copy
import spectral
import time
import csv
import os
import importlib
import pickle
import utils
from hsiViewer import hsi_viewer_layers as hlv
from hsiViewer import hsi_viewer_ROI as hvr
import matplotlib as mpl
mpl.rcParams['lines.linewidth'] = 0.75

# --- Load configuration (paths + parameters live in config.yaml) ---
import yaml
with open('config.yaml') as _f:
    CONFIG = yaml.safe_load(_f)

**Load the calibration** (`gain`, `offset`) produced by notebook 01.

In [ ]:
# Read the gain and offset
gain = np.load(CONFIG['paths']['gain'])
offset = np.load(CONFIG['paths']['offset'])

## Open the Image to Convert to Reflectance

**Open a raw image to convert**, and read the smoothing level and bad-band ranges from the config. Set the image in `config.yaml`.

In [ ]:
# number of smoothing iterations
smoothing_level = CONFIG['reflectance']['smoothing_level']
# wl range(s) to remove (in nanometers)
bbl_wl_ranges = CONFIG['reflectance']['bbl_wl_ranges']

# raw image to convert to reflectance
dir = CONFIG['paths']['raw_image_dir']
fname = CONFIG['paths']['raw_image']
fname_hdr = CONFIG['paths']['raw_image'] + '.hdr'
# Read the image
im = spectral.envi.open(os.path.join(dir,fname_hdr), os.path.join(dir,fname))
#im.Arr = im.load().astype(np.float32)
#im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

## Convert Image to Reflectance

**Convert to reflectance.** Drops bad bands, applies per-band gain/offset, masks empty pixels, spatially smooths, and saves the reflectance image next to the raw one.

In [ ]:
# ====== Convert to Reflectance ======
# wl range(s) to remove (in nanometers)
bbl_wl_ranges = CONFIG['reflectance']['bbl_wl_ranges']
indices = []
for i in range(len(wl)):
    is_bad_band = False
    for bbl_wl_range in bbl_wl_ranges:
        if bbl_wl_range[0] < wl[i] < bbl_wl_range[1]:
            is_bad_band = True                
    if (not is_bad_band):
        indices.append(int(i))
indices = np.asarray(indices, dtype=np.int16)

# determine the parameters for the image
nr = im.nrows
nc = im.ncols
nb = len(indices)

# subset the gain and offset to the good bands
gain = gain[indices]
offset = offset[indices]
wl = wl[indices]

# Prepare an output array for the reflectance image
imRef = np.zeros((nr, nc, nb), dtype=np.float32)
# Create the data mask
mask = (im.read_band(0) > 0).astype(np.float32)
    
# ====== Load the image and compute reflectance ======
# Loop over bands and fill the result
print('Reading the image and converting to reflectance.')
for i, b in enumerate(indices):
    imRef[:, :, i] = (gain[i]*np.squeeze(im.read_band(b) + offset[i])*mask).astype(np.float32)
                            
# ====== Spatially smooth the image ======
for i in range(smoothing_level):
    print(f'Smoothing the image, iteration {i+1}.')
    imRef = utils.spatial_smoothing(imRef, mask=mask).astype(np.float32)

# ====== Save the image ======
# Save the image
print('Saving the image.')
md=im.metadata
md['wavelength'] = [str(w) for w in wl]
spectral.envi.save_image(os.path.join(dir,fname+'_ref.hdr'), imRef, metadata=md, force=True)

**Optional — open a reflectance image** to inspect it.

In [ ]:
# data directory
dir = CONFIG['paths']['reflectance_image_dir']
fname = CONFIG['paths']['reflectance_image']
fname_hdr = CONFIG['paths']['reflectance_image'].rsplit('.', 1)[0] + '.hdr'
# Read the image
im = spectral.envi.open(os.path.join(dir,fname_hdr), os.path.join(dir,fname))
im.Arr = im.load().astype(np.float32)
im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

**Interactive.** Opens the reflectance image in the hsiViewer to examine pixels and spectra.

In [ ]:
# If you want to manually examine the image and sepctra
hlv.viewer(im)